In [0]:
%pip install spark

In [0]:
df = spark.read.option("header", True).option("inferSchema", True).csv("/Volumes/workspace/default/home_credit/application_train.csv")

In [0]:
display(df)

In [0]:
display(df.describe())

In [0]:
print(f"El DataFrame tiene {len(df.columns)} columnas.")


In [0]:
df.groupBy('CNT_CHILDREN').count().orderBy('CNT_CHILDREN').show() 

In [0]:
df.filter(df.CNT_CHILDREN >= 14).select('CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'AMT_INCOME_TOTAL').show()

Después de una revisión a la variable CNT_CHILDREN, se decidió mantener los valores que se consideran outliers dentro del dataset de entrenamiento, puesto que se va a usar el modelo LightGBM y para este modelo no representa importancia valores extremos, como en este caso familias con más de 14 hijos o 20 miembros de la familia, ya que se encarga de clasificar y esto no modifica su funcionamiento.

In [0]:
df.groupBy("TARGET").count().show()

In [0]:
df.createOrReplaceTempView("df")

In [0]:
%sql

SELECT
    TARGET, 
    count(TARGET),
    (COUNT(TARGET) * 100) / SUM(count(TARGET)) OVER() as porcentaje_total
FROM df
GROUP BY TARGET

In [0]:
pos = 24825
neg = 282686

scale = neg/pos
print(f"La cantidad de clientes que no pagan es {neg} y la cantidad de clientes que pagan es {pos}")
print(f"El factor de escala es {scale}")

La cantidad de clientes que no pagan es 282686 y la cantidad de clientes que pagan es 24825
El factor de escala es 11.387150050352467

In [0]:
df_boxplo = df.select('TARGET', "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3").toPandas()

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

pdf_long = df_boxplo.melt(id_vars='TARGET', value_vars=['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'],
                     var_name='variable', value_name='puntaje')

plt.figure(figsize=(10, 6))
sns.boxplot(data=pdf_long, x='variable', y='puntaje', hue='TARGET')
plt.xticks([0, 1, 2], ['Score 1', 'Score 2', 'Score 3'])
plt.ylabel('Puntaje')
plt.title('Score externo 1/2/3 por TARGET')
plt.show()

In [0]:
df_boxplo.groupby('TARGET')[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].describe().transpose()

**NOTA:** Se muestra como las personas que si pagan tienen valñores de puntuaje externos mucho mása altos, lo que es esperable, ya ue tiene coherencia, además cuetnan con medianas más altas en los diferents puntajes externos.

**NOTA:** Los outlier de EXT_SOURCE_2 de las personas que no pagan, ya que son personas que a pesar de tener un puntuaje bastante alto, lo que signficaría que son confiables, igualmente dieron por default, así que EXT_SOURCE es una beuna variable apra el atregt, pero se encesita apoyo de otras más. 

In [0]:
pdf2 = df.select('TARGET', 'AMT_INCOME_TOTAL').toPandas()

plt.figure(figsize=(8, 6))
sns.boxplot(data=pdf2, x='TARGET', y='AMT_INCOME_TOTAL')
plt.xticks([0, 1], ['No Paga', 'Paga'])
plt.ylabel('Ingreso anual')
plt.title('Ingreso anual por TARGET')
plt.show()

In [0]:
pdf2.groupby('TARGET')['AMT_INCOME_TOTAL'].describe().transpose()

**NOTA:** No hay mucha difrencia en las medias, las cajas se solapan mucho en el mismo lugar, así que sola no será un variable muy relevante, pero será importante para el **debt-to-income ratio**

In [0]:
df.select('DAYS_EMPLOYED').summary().show()

In [0]:
df.filter(df.DAYS_EMPLOYED == 365243).count()

La variable `DAYS_EMPLOYED` cuenta con valores negativos, ya que representa los días trabajados por el cliente hasta el día de solicitar el crédito. Hay registros con un mismo valor positivo, que es 365243, que es una manera de decir que esa persona no cuenta con días trabajados hasta la solicitud del crédito.